In [1]:
# 1. Import Libraries

from google.colab import drive
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping



In [3]:
# 2. Mount Drive and Load Dataset

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/weather_data_only.csv'
df = pd.read_csv(file_path)

print("Dataset loaded:", df.shape)


Mounted at /content/drive
Dataset loaded: (189888, 6)


In [4]:
# 3. Preprocessing

df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Time features
df['hour'] = df['Timestamp'].dt.hour
df['day'] = df['Timestamp'].dt.day
df['month'] = df['Timestamp'].dt.month
df['dayofweek'] = df['Timestamp'].dt.dayofweek

# cyclical encoding (better than raw hour/month)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Features
feature_cols = [
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dayofweek'
]

# Targets
target_cols = [
    'Temperature (°C)', 'Humidity (%)',
    'Wind Speed (m/s)', 'Rainfall (mm)',
    'Solar Irradiance (W/m²)'
]

X = df[feature_cols]
y = df[target_cols]


In [5]:
# 4. Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print("Train:", len(X_train))
print("Test :", len(X_test))

Train: 151910
Test : 37978


In [6]:
# 5. Scaling

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [7]:
# 6. Build Better Model

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(y_train.shape[1])
])

model.compile(optimizer='adam', loss='mse')

# prevents overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
# 7. Train Model

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - loss: 1136.4840 - val_loss: 519.6741
Epoch 2/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 514.2197 - val_loss: 514.7713
Epoch 3/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 514.6190 - val_loss: 513.1412
Epoch 4/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 514.5719 - val_loss: 513.1124
Epoch 5/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 513.8935 - val_loss: 512.5730
Epoch 6/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 514.1762 - val_loss: 512.8422
Epoch 7/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 514.4322 - val_loss: 515.2711
Epoch 8/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 514.3353 - val_loss: 513.6602
Epoch 9/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 514.1914 - val_loss: 513.4255
Epoch 10/50
1899/1899 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 514.1695 - val_loss: 518.5770


In [9]:
# 8. Predictions

y_pred = model.predict(X_test)


1187/1187 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [10]:
# 9. Regression Metrics

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- Regression Metrics ---")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R2   :", r2)



--- Regression Metrics ---
MAE  : 9.981605529785156
RMSE : 22.63091525315122
R2   : -0.012661385349929333


In [11]:
# 10. Classification


# Rainfall column
y_test_rain = y_test['Rainfall (mm)'].values
y_pred_rain = y_pred[:, 3]

# Better threshold
threshold = 0.5

y_test_bin = (y_test_rain > threshold).astype(int)
y_pred_bin = (y_pred_rain > threshold).astype(int)

# Check class distribution
print("\nRain class distribution:", np.bincount(y_test_bin))

if len(np.unique(y_test_bin)) > 1:
    accuracy = accuracy_score(y_test_bin, y_pred_bin)
    precision = precision_score(y_test_bin, y_pred_bin, zero_division=0)
    recall = recall_score(y_test_bin, y_pred_bin, zero_division=0)
    f1 = f1_score(y_test_bin, y_pred_bin, zero_division=0)
    roc_auc = roc_auc_score(y_test_bin, y_pred_rain)

    print("\n--- Classification Metrics ---")
    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
    print("ROC AUC  :", roc_auc)
else:
    print("\n⚠️ Only one class present — classification metrics not reliable.")



Rain class distribution: [ 3628 34350]

--- Classification Metrics ---
Accuracy : 0.9044710095318342
Precision: 0.9044710095318342
Recall   : 1.0
F1 Score : 0.9498396195111161
ROC AUC  : 0.4940956237191245


In [12]:
# 11. Save Model
model.save('weather_dense_model.keras')
print("\nModel saved as weather_dense_model.keras")



Model saved as weather_dense_model.keras


In [13]:
# 12. Prediction Function

def predict_weather(timestamp_str):
    dt = pd.to_datetime(timestamp_str)

    features = pd.DataFrame([[
        np.sin(2 * np.pi * dt.hour / 24),
        np.cos(2 * np.pi * dt.hour / 24),
        np.sin(2 * np.pi * dt.month / 12),
        np.cos(2 * np.pi * dt.month / 12),
        dt.dayofweek
    ]], columns=feature_cols)

    features_scaled = scaler.transform(features)

    pred = model.predict(features_scaled)[0]
    return dict(zip(target_cols, pred))


In [14]:
# 13. Example Prediction

print("\n--- Example Prediction ---")
example = predict_weather('2020-07-01 14:00')

for k, v in example.items():
    print(f"{k}: {v:.2f}")


--- Example Prediction ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Temperature (°C): 28.28
Humidity (%): 79.43
Wind Speed (m/s): 2.18
Rainfall (mm): 4.55
Solar Irradiance (W/m²): 251.80


In [15]:
# 14. User Input

user_input = input("\nEnter timestamp (e.g., 2020-08-15 10:30): ")

try:
    result = predict_weather(user_input)
    print("\nPredicted Weather:")
    for k, v in result.items():
        print(f"{k}: {v:.2f}")
except Exception as e:
    print("Error:", e)


Enter timestamp (e.g., 2020-08-15 10:30): 2027-10-09   5:10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

Predicted Weather:
Temperature (°C): 28.16
Humidity (%): 79.47
Wind Speed (m/s): 2.13
Rainfall (mm): 4.46
Solar Irradiance (W/m²): 251.49
